In [1]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
from rasterio.enums import Resampling

# Set matplotlib config directory to avoid permission issues
os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

print("✓ Libraries imported successfully")
print(f"  rasterio version: {rasterio.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  rasterio version: 1.5.0
  NumPy version: 1.26.4


In [2]:
# CORINE class descriptions (simplified codes 1-44 used in raster)
CORINE_CLASSES = {
    # Artificial surfaces (1-11)
    1: "Continuous urban fabric",
    2: "Discontinuous urban fabric",
    3: "Industrial or commercial units",
    4: "Road and rail networks",
    5: "Port areas",
    6: "Airports",
    7: "Mineral extraction sites",
    8: "Dump sites",
    9: "Construction sites",
    10: "Green urban areas",
    11: "Sport and leisure facilities",
    
    # Agricultural areas (12-22)
    12: "Non-irrigated arable land",
    13: "Permanently irrigated land",
    14: "Rice fields",
    15: "Vineyards",
    16: "Fruit trees and berry plantations",
    17: "Olive groves",
    18: "Pastures",
    19: "Annual crops with permanent crops",
    20: "Complex cultivation patterns",
    21: "Agriculture with natural vegetation",
    22: "Agro-forestry areas",
    
    # Forest and semi-natural areas (23-34)
    23: "Broad-leaved forest",
    24: "Coniferous forest",
    25: "Mixed forest",
    26: "Natural grasslands",
    27: "Moors and heathland",
    28: "Sclerophyllous vegetation",
    29: "Transitional woodland-shrub",
    30: "Beaches, dunes, sands",
    31: "Bare rocks",
    32: "Sparsely vegetated areas",
    33: "Burnt areas",
    34: "Glaciers and perpetual snow",
    
    # Wetlands (35-39)
    35: "Inland marshes",
    36: "Peat bogs",
    37: "Salt marshes",
    38: "Salines",
    39: "Intertidal flats",
    
    # Water bodies (40-44)
    40: "Water courses",
    41: "Water bodies",
    42: "Coastal lagoons",
    43: "Estuaries",
    44: "Sea and ocean",
    
    48: "No data"
}

# Color mapping for visualization
CORINE_COLORS = {
    1: '#E6004D', 2: '#FF0000', 3: '#CC4DF2', 4: '#CC0000', 5: '#E6CCCC',
    6: '#E6CCE6', 7: '#A600CC', 8: '#A64DCC', 9: '#FF4DFF', 10: '#FFA6FF',
    11: '#FFE6FF', 12: '#FFFFA8', 13: '#FFFF00', 14: '#E6E600', 15: '#E68000',
    16: '#F2A64D', 17: '#E6A600', 18: '#E6E64D', 19: '#FFE6A6', 20: '#FFE64D',
    21: '#E6CC4D', 22: '#F2CCA6', 23: '#80FF00', 24: '#00A600', 25: '#4DFF00',
    26: '#CCF24D', 27: '#A6FF80', 28: '#A6E64D', 29: '#A6F200', 30: '#E6E6E6',
    31: '#CCCCCC', 32: '#CCFFCC', 33: '#000000', 34: '#A6E6CC', 35: '#A6A6FF',
    36: '#4D4DFF', 37: '#CCCCFF', 38: '#E6E6FF', 39: '#A6A6E6', 40: '#00CCF2',
    41: '#80F2E6', 42: '#00FFA6', 43: '#A6FFE6', 44: '#E6F2FF'
}

print(f"✓ Defined {len(CORINE_CLASSES)} CORINE land cover classes")

✓ Defined 45 CORINE land cover classes


In [6]:
# ============================================================
# CONFIGURATION - Edit these values!
# ============================================================

# Get username for default paths
USER = os.environ.get('USER', 'user')

# --- Path Configuration ---
# Directory containing your downloaded Sentinel-2 .SAFE folders
S2_DATA_DIR = f"/p/scratch/training2600/TeamGudnason/data"

# CORINE Land Cover raster (shared location)
CORINE_PATH = "/p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif"

# Output directory for processed data (stacked bands + aligned CORINE)
#OUTPUT_DIR = f"/p/scratch/training2600/TeamGudnason/data/aligned_data
OUTPUT_DIR = "/p/scratch/training2600/TeamGudnason/training_data/final_geo"

# --- Tile Selection ---
# Options:
#   'all'  - Process all .SAFE directories found in S2_DATA_DIR
#   'list' - Process only tiles listed in TILE_LIST below
TILE_SELECTION = 'all'

# If TILE_SELECTION = 'list', specify which tiles to process:
TILE_LIST = [
    # Add your specific tile names here, e.g.:
    # "S2A_MSIL2A_20181019T102031_N0500_R065_T33UUP_20230813T105225.SAFE",
]

# --- Processing Options ---
SKIP_EXISTING = False  # Skip tiles that already have output files

# ============================================================
print("Configuration:")
print(f"  S2 Data Directory: {S2_DATA_DIR}")
print(f"  CORINE Path: {CORINE_PATH}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile Selection: {TILE_SELECTION}")

Configuration:
  S2 Data Directory: /p/scratch/training2600/TeamGudnason/data
  CORINE Path: /p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif
  Output Directory: /p/scratch/training2600/TeamGudnason/training_data/final_geo
  Tile Selection: all


In [7]:
def find_s2_tiles(data_dir, selection='all', tile_list=None):
    """
    Find Sentinel-2 .SAFE directories to process.
    
    Parameters:
    -----------
    data_dir : str
        Directory containing .SAFE folders
    selection : str
        'all' to find all tiles, 'list' to use tile_list
    tile_list : list
        List of specific tile names to process
    
    Returns:
    --------
    list : List of Path objects for .SAFE directories
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        return []
    
    if selection == 'all':
        tiles = sorted(data_path.glob("*.SAFE"))
    else:
        tiles = []
        for tile_name in (tile_list or []):
            tile_path = data_path / tile_name
            if tile_path.exists():
                tiles.append(tile_path)
            else:
                print(f"⚠ Tile not found: {tile_name}")
    
    return tiles

# Find tiles
available_tiles = find_s2_tiles(S2_DATA_DIR, TILE_SELECTION, TILE_LIST)

print(f"\n📂 Found {len(available_tiles)} Sentinel-2 tile(s) to process:")
for i, tile in enumerate(available_tiles, 1):
    print(f"  {i}. {tile.name}")

if len(available_tiles) == 0:
    print("\n⚠ No tiles found! Check your S2_DATA_DIR path.")


📂 Found 4 Sentinel-2 tile(s) to process:
  1. S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859.SAFE
  2. S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408.SAFE
  3. S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829.SAFE
  4. S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012.SAFE


In [8]:
# Check CORINE file exists and get basic info
corine_path = Path(CORINE_PATH)

if not corine_path.exists():
    print(f"❌ CORINE file not found: {CORINE_PATH}")
    print("\nPlease check the path or download CORINE data.")
else:
    with rasterio.open(corine_path) as src:
        print("✓ CORINE Land Cover 2018")
        print(f"  File: {corine_path.name}")
        print(f"  Size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  NoData value: {src.nodata}")
        print(f"  Bounds: {src.bounds}")

✓ CORINE Land Cover 2018
  File: U2018_CLC2018_V2020_20u1.tif
  Size: 65000 x 46000 pixels
  Resolution: 100.0m x 100.0m
  CRS: PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3035"]]
  NoData value: -128.0
  Bounds: BoundingBox(left=900000.0, bottom=900000.0, right=7400000.0, top=5500000.0)


In [9]:
def stack_s2_bands(safe_path, output_path):
    """
    Stack Sentinel-2 bands (B02, B03, B04, B08) into a single GeoTIFF.
    
    Parameters:
    -----------
    safe_path : Path
        Path to .SAFE directory
    output_path : Path
        Output path for stacked GeoTIFF
    
    Returns:
    --------
    dict : Info about the stacked file (crs, bounds, etc.) or None on failure
    """
    bands = ['B02', 'B03', 'B04', 'B08']  # Blue, Green, Red, NIR
    resolution = '10m'
    
    try:
        # Find band files
        band_paths = {}
        for band_name in bands:
            pattern = f"**/R{resolution}/*_{band_name}_{resolution}.jp2"
            matches = list(safe_path.glob(pattern))
            if matches:
                band_paths[band_name] = matches[0]
            else:
                print(f"  ⚠ {band_name} not found")
                return None
        
        # Read first band for metadata
        with rasterio.open(band_paths['B02']) as src:
            profile = src.profile.copy()
            height, width = src.height, src.width
            crs = src.crs
            transform = src.transform
            bounds = src.bounds
        
        # Update profile for output
        profile.update(
            driver='GTiff',
            count=len(bands),
            dtype='uint16',
            compress='lzw',
            tiled=True,
            blockxsize=256,
            blockysize=256
        )
        
        # Create output and write bands
        with rasterio.open(output_path, 'w', **profile) as dst:
            for i, (band_name, band_path) in enumerate(band_paths.items(), start=1):
                with rasterio.open(band_path) as src:
                    data = src.read(1)
                    dst.write(data, i)
                    dst.set_band_description(i, band_name)
        
        return {
            'crs': crs,
            'width': width,
            'height': height,
            'transform': transform,
            'bounds': bounds,
            'bands': bands
        }
        
    except Exception as e:
        print(f"  ❌ Band stacking failed: {e}")
        import traceback
        traceback.print_exc()
        return None


print("✓ Band stacking function defined")

✓ Band stacking function defined


In [10]:
def align_corine_to_s2(corine_path, s2_path, output_path):
    """
    Align CORINE raster to match Sentinel-2 tile geometry using rasterio.
    
    Parameters:
    -----------
    corine_path : Path
        Path to CORINE GeoTIFF
    s2_path : Path
        Path to stacked S2 GeoTIFF (reference)
    output_path : Path
        Output path for aligned CORINE
    
    Returns:
    --------
    bool : Success status
    """
    try:
        # Get S2 geometry (target)
        with rasterio.open(s2_path) as s2_src:
            dst_crs = s2_src.crs
            dst_transform = s2_src.transform
            dst_width = s2_src.width
            dst_height = s2_src.height
            dst_bounds = s2_src.bounds
        
        print(f"    Target CRS: {dst_crs}")
        print(f"    Target size: {dst_width} x {dst_height} pixels")
        print(f"    Target bounds: {dst_bounds}")
        
        # Open CORINE and reproject
        with rasterio.open(corine_path) as src:
            # Create output profile
            out_profile = src.profile.copy()
            out_profile.update(
                driver='GTiff',
                crs=dst_crs,
                transform=dst_transform,
                width=dst_width,
                height=dst_height,
                compress='lzw',
                tiled=True
            )
            
            # Create output raster
            with rasterio.open(output_path, 'w', **out_profile) as dst:
                # Reproject using nearest neighbor (for categorical data)
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=dst_transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest
                )
        
        return True
        
    except Exception as e:
        print(f"  ❌ CORINE alignment failed: {e}")
        import traceback
        traceback.print_exc()
        return False


def check_corine_coverage(corine_path):
    """
    Check if aligned CORINE has valid data.
    
    Returns:
    --------
    tuple : (has_valid_data, unique_classes, coverage_percent)
    """
    with rasterio.open(corine_path) as src:
        data = src.read(1)
    
    total_pixels = data.size
    valid_mask = (data >= 1) & (data <= 44)
    valid_pixels = np.sum(valid_mask)
    coverage_percent = (valid_pixels / total_pixels) * 100
    
    unique = np.unique(data[valid_mask])
    
    return len(unique) > 0, unique, coverage_percent


print("✓ CORINE alignment functions defined")

✓ CORINE alignment functions defined


In [11]:
def process_tile_alignment(safe_path, corine_path, output_dir, skip_existing=True):
    """
    Process a single Sentinel-2 tile: stack bands and align CORINE.
    
    Returns:
    --------
    dict : Processing results
    """
    tile_name = safe_path.stem
    result = {
        'tile': tile_name,
        'status': 'unknown',
        's2_stacked': None,
        'corine_aligned': None,
        'crs': None,
        'n_classes': 0,
        'coverage_percent': 0
    }
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Output paths
    s2_stacked = output_dir / f"{tile_name}_stacked.tif"
    corine_aligned = output_dir / f"corine_aligned_{tile_name}.tif"
    
    # Check for existing output
    if skip_existing and s2_stacked.exists() and corine_aligned.exists():
        print(f"  ⏭ Skipping (outputs exist)")
        has_coverage, classes, coverage = check_corine_coverage(corine_aligned)
        result['status'] = 'skipped'
        result['s2_stacked'] = str(s2_stacked)
        result['corine_aligned'] = str(corine_aligned)
        result['n_classes'] = len(classes)
        result['coverage_percent'] = coverage
        return result
    
    # Step 1: Stack bands
    print(f"  📦 Stacking S2 bands...")
    stack_info = stack_s2_bands(safe_path, s2_stacked)
    if stack_info is None:
        result['status'] = 'failed_stacking'
        return result
    
    result['crs'] = str(stack_info['crs'])
    print(f"    Created: {s2_stacked.name}")
    print(f"    Size: {stack_info['width']} x {stack_info['height']} pixels")
    print(f"    CRS: {stack_info['crs']}")
    
    # Step 2: Align CORINE
    print(f"  🗺️ Aligning CORINE to S2 geometry...")
    if not align_corine_to_s2(corine_path, s2_stacked, corine_aligned):
        result['status'] = 'failed_alignment'
        return result
    
    # Step 3: Check coverage
    print(f"  🔍 Checking CORINE coverage...")
    has_coverage, classes, coverage = check_corine_coverage(corine_aligned)
    
    if not has_coverage:
        print(f"  ⚠ No valid CORINE data (tile outside coverage?)")
        result['status'] = 'no_coverage'
        return result
    
    result['status'] = 'success'
    result['s2_stacked'] = str(s2_stacked)
    result['corine_aligned'] = str(corine_aligned)
    result['n_classes'] = len(classes)
    result['coverage_percent'] = coverage
    
    print(f"    CORINE classes found: {len(classes)}")
    print(f"    Valid coverage: {coverage:.1f}%")
    
    return result


print("✓ Tile processing function defined")

✓ Tile processing function defined


In [12]:
# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

print("="*70)
print("ALIGNING CORINE WITH SENTINEL-2 TILES")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tiles to process: {len(available_tiles)}")
print(f"Output directory: {OUTPUT_DIR}")
print()

# Store results
all_results = []

# Process each tile
for i, tile_path in enumerate(available_tiles, 1):
    print(f"\n[{i}/{len(available_tiles)}] Processing: {tile_path.name}")
    
    result = process_tile_alignment(
        safe_path=tile_path,
        corine_path=CORINE_PATH,
        output_dir=OUTPUT_DIR,
        skip_existing=SKIP_EXISTING
    )
    
    all_results.append(result)
    
    if result['status'] == 'success':
        print(f"  ✅ Success: {result['n_classes']} classes, {result['coverage_percent']:.1f}% coverage")
    elif result['status'] == 'skipped':
        print(f"  ⏭ Skipped (already processed)")
    else:
        print(f"  ❌ Status: {result['status']}")

# Summary
print("\n" + "="*70)
print("ALIGNMENT SUMMARY")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

successful = [r for r in all_results if r['status'] == 'success']
skipped = [r for r in all_results if r['status'] == 'skipped']
failed = [r for r in all_results if r['status'] not in ['success', 'skipped']]

print(f"\nResults:")
print(f"  ✅ Successful: {len(successful)}")
print(f"  ⏭ Skipped: {len(skipped)}")
print(f"  ❌ Failed: {len(failed)}")
print(f"\n  Output directory: {OUTPUT_DIR}")

ALIGNING CORINE WITH SENTINEL-2 TILES
Start time: 2026-04-19 19:04:28
Tiles to process: 4
Output directory: /p/scratch/training2600/TeamGudnason/training_data/final_geo


[1/4] Processing: S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859.SAFE
  📦 Stacking S2 bands...
    Created: S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_stacked.tif
    Size: 10980 x 10980 pixels
    CRS: EPSG:32633
  🗺️ Aligning CORINE to S2 geometry...
    Target CRS: EPSG:32633
    Target size: 10980 x 10980 pixels
    Target bounds: BoundingBox(left=699960.0, bottom=5190240.0, right=809760.0, top=5300040.0)
  🔍 Checking CORINE coverage...
    CORINE classes found: 26
    Valid coverage: 100.0%
  ✅ Success: 26 classes, 100.0% coverage

[2/4] Processing: S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408.SAFE
  📦 Stacking S2 bands...
    Created: S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408_stacked.tif
    Size: 10980 x 10980 pixels
    CRS: EPSG:32633
  🗺️ Ali

In [18]:
def visualize_alignment(s2_path, corine_path, title="Alignment Verification", save_path=None):
    """
    Visualize the S2 image and aligned CORINE side by side.
    
    Parameters:
    -----------
    s2_path : str or Path
        Path to stacked S2 GeoTIFF
    corine_path : str or Path
        Path to aligned CORINE GeoTIFF
    title : str
        Plot title
    save_path : str or Path, optional
        Path to save the figure (PNG format)
    """
    # Load S2 data
    with rasterio.open(s2_path) as src:
        # Read RGB bands (B04, B03, B02 = bands 3, 2, 1)
        r = src.read(3).astype(float)  # B04 (Red)
        g = src.read(2).astype(float)  # B03 (Green)
        b = src.read(1).astype(float)  # B02 (Blue)
    
    # Create RGB composite
    rgb = np.stack([r, g, b], axis=-1)
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
    
    # Load CORINE data
    with rasterio.open(corine_path) as src:
        corine = src.read(1)
    
    # Create CORINE colormap
    from matplotlib.colors import ListedColormap
    unique_classes = np.unique(corine[(corine >= 1) & (corine <= 44)])
    colors = [CORINE_COLORS.get(c, '#888888') for c in range(45)]
    cmap = ListedColormap(colors)
    
    # Create figure (subsample for display)
    step = 10
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # S2 RGB
    axes[0].imshow(rgb[::step, ::step])
    axes[0].set_title('Sentinel-2 RGB (B04, B03, B02)', fontsize=12)
    axes[0].axis('off')
    
    # CORINE
    axes[1].imshow(corine[::step, ::step], cmap=cmap, vmin=0, vmax=44)
    axes[1].set_title('Aligned CORINE Land Cover', fontsize=12)
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(rgb[::step, ::step])
    corine_masked = np.ma.masked_where((corine < 1) | (corine > 44), corine)
    axes[2].imshow(corine_masked[::step, ::step], cmap=cmap, vmin=0, vmax=44, alpha=0.4)
    axes[2].set_title('S2 + CORINE Overlay', fontsize=12)
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save figure if path provided
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"✓ Figure saved to: {save_path}")
    
    plt.show()
    
    # Print class distribution
    print("\nCORINE Classes in this tile:")
    for c in sorted(unique_classes):
        count = np.sum(corine == c)
        pct = count / corine.size * 100
        print(f"  Class {c:2d}: {CORINE_CLASSES.get(c, 'Unknown')[:30]:30s} - {count:,} pixels ({pct:.1f}%)")


# Results directory for saving figures
RESULTS_DIR = f"/p/project1/training2600/TeamGudnason/results"

# Visualize first successful result
successful_results = [r for r in all_results if r['status'] in ['success', 'skipped']]

if successful_results:
    result = successful_results[2]
    tile_name = result['tile']
    
    # Create save path for the visualization
    save_path = Path(RESULTS_DIR) / f"alignment_{tile_name[:50]}.png"
    
    print(f"Visualizing: {tile_name[:50]}...")
    visualize_alignment(
        result['s2_stacked'], 
        result['corine_aligned'],
        title=f"Alignment: {tile_name[:40]}...",
        save_path=save_path
    )
else:
    print("No successful results to visualize.")

Visualizing: S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230...
✓ Figure saved to: /p/project1/training2600/TeamGudnason/results/alignment_S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230.png

CORINE Classes in this tile:
  Class  1: Continuous urban fabric        - 139,776 pixels (0.1%)
  Class  2: Discontinuous urban fabric     - 8,193,939 pixels (6.8%)
  Class  3: Industrial or commercial units - 2,067,603 pixels (1.7%)
  Class  4: Road and rail networks         - 303,675 pixels (0.3%)
  Class  5: Port areas                     - 35,440 pixels (0.0%)
  Class  6: Airports                       - 191,065 pixels (0.2%)
  Class  7: Mineral extraction sites       - 220,399 pixels (0.2%)
  Class  8: Dump sites                     - 84,967 pixels (0.1%)
  Class  9: Construction sites             - 58,566 pixels (0.0%)
  Class 10: Green urban areas              - 205,195 pixels (0.2%)
  Class 11: Sport and leisure facilities   - 1,513,782 pixels (1.3%)
  Class 12: Non-irrigated arable 

In [19]:
# Display detailed results
print("Detailed Results per Tile:")
print("-" * 90)
print(f"{'Tile':<50} {'Status':<15} {'Classes':<10} {'Coverage':<10}")
print("-" * 90)

for result in all_results:
    tile_short = result['tile'][:48] + ".." if len(result['tile']) > 50 else result['tile']
    coverage = f"{result['coverage_percent']:.1f}%" if result['coverage_percent'] > 0 else "-"
    classes = str(result['n_classes']) if result['n_classes'] > 0 else "-"
    print(f"{tile_short:<50} {result['status']:<15} {classes:<10} {coverage:<10}")

print("-" * 90)

# List output files
print("\n📁 Output Files:")
output_path = Path(OUTPUT_DIR)
if output_path.exists():
    for f in sorted(output_path.glob("*.tif")):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.1f} MB)")
else:
    print(f"  Output directory not found: {OUTPUT_DIR}")

Detailed Results per Tile:
------------------------------------------------------------------------------------------
Tile                                               Status          Classes    Coverage  
------------------------------------------------------------------------------------------
S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_202.. success         26         100.0%    
S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_202.. success         26         100.0%    
S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_202.. success         26         100.0%    
S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_202.. success         26         100.0%    
------------------------------------------------------------------------------------------

📁 Output Files:
  S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_stacked.tif (2252.1 MB)
  S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408_stacked.tif (2218.9 MB)
  S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829_s

In [ ]:
## lab 4.2

In [20]:
# CORINE class descriptions (simplified codes 1-44 used in raster)
CORINE_CLASSES = {
    # Artificial surfaces (1-11)
    1: "Continuous urban fabric",
    2: "Discontinuous urban fabric",
    3: "Industrial or commercial units",
    4: "Road and rail networks",
    5: "Port areas",
    6: "Airports",
    7: "Mineral extraction sites",
    8: "Dump sites",
    9: "Construction sites",
    10: "Green urban areas",
    11: "Sport and leisure facilities",
    
    # Agricultural areas (12-22)
    12: "Non-irrigated arable land",
    13: "Permanently irrigated land",
    14: "Rice fields",
    15: "Vineyards",
    16: "Fruit trees and berry plantations",
    17: "Olive groves",
    18: "Pastures",
    19: "Annual crops with permanent crops",
    20: "Complex cultivation patterns",
    21: "Agriculture with natural vegetation",
    22: "Agro-forestry areas",
    
    # Forest and semi-natural areas (23-34)
    23: "Broad-leaved forest",
    24: "Coniferous forest",
    25: "Mixed forest",
    26: "Natural grasslands",
    27: "Moors and heathland",
    28: "Sclerophyllous vegetation",
    29: "Transitional woodland-shrub",
    30: "Beaches, dunes, sands",
    31: "Bare rocks",
    32: "Sparsely vegetated areas",
    33: "Burnt areas",
    34: "Glaciers and perpetual snow",
    
    # Wetlands (35-39)
    35: "Inland marshes",
    36: "Peat bogs",
    37: "Salt marshes",
    38: "Salines",
    39: "Intertidal flats",
    
    # Water bodies (40-44)
    40: "Water courses",
    41: "Water bodies",
    42: "Coastal lagoons",
    43: "Estuaries",
    44: "Sea and ocean",
    
    48: "No data"
}

# Color mapping for visualization
CORINE_COLORS = {
    1: '#E6004D', 2: '#FF0000', 3: '#CC4DF2', 4: '#CC0000', 5: '#E6CCCC',
    6: '#E6CCE6', 7: '#A600CC', 8: '#A64DCC', 9: '#FF4DFF', 10: '#FFA6FF',
    11: '#FFE6FF', 12: '#FFFFA8', 13: '#FFFF00', 14: '#E6E600', 15: '#E68000',
    16: '#F2A64D', 17: '#E6A600', 18: '#E6E64D', 19: '#FFE6A6', 20: '#FFE64D',
    21: '#E6CC4D', 22: '#F2CCA6', 23: '#80FF00', 24: '#00A600', 25: '#4DFF00',
    26: '#CCF24D', 27: '#A6FF80', 28: '#A6E64D', 29: '#A6F200', 30: '#E6E6E6',
    31: '#CCCCCC', 32: '#CCFFCC', 33: '#000000', 34: '#A6E6CC', 35: '#A6A6FF',
    36: '#4D4DFF', 37: '#CCCCFF', 38: '#E6E6FF', 39: '#A6A6E6', 40: '#00CCF2',
    41: '#80F2E6', 42: '#00FFA6', 43: '#A6FFE6', 44: '#E6F2FF'
}

print(f"✓ Defined {len(CORINE_CLASSES)} CORINE land cover classes")

✓ Defined 45 CORINE land cover classes


In [21]:
# --- Path Configuration ---
# Directory containing aligned data from Lab 4.1
ALIGNED_DATA_DIR = f"/p/scratch/training2600/TeamGudnason/data/aligned_data"

# Output directory for training patches
OUTPUT_DIR = f"/p/scratch/training2600/TeamGudnason/training_data"

# --- Patch Extraction Parameters ---
PATCH_SIZE = 3       # Size in pixels (3 = 30m x 30m at 10m resolution)
STRIDE = None        # Stride for extraction (None = same as PATCH_SIZE, no overlap)
MAX_PATCHES = 50000  # Maximum patches per tile

# --- Normalization Settings ---
# Options: 'minmax', 'zscore', 'percentile', 'none'
NORMALIZATION = 'percentile'

# For percentile normalization: clip to these percentiles
PERCENTILE_LOW = 2
PERCENTILE_HIGH = 98

# --- Processing Options ---
SKIP_EXISTING = True  # Skip tiles that already have output files

# ============================================================
print("Configuration:")
print(f"  Aligned Data Directory: {ALIGNED_DATA_DIR}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile Selection: {TILE_SELECTION}")
print(f"  Patch Size: {PATCH_SIZE}x{PATCH_SIZE} ({PATCH_SIZE * 10}m x {PATCH_SIZE * 10}m)")
print(f"  Max Patches per Tile: {MAX_PATCHES:,}")
print(f"  Normalization: {NORMALIZATION}")

Configuration:
  Aligned Data Directory: /p/scratch/training2600/TeamGudnason/data/aligned_data
  Output Directory: /p/scratch/training2600/TeamGudnason/training_data
  Tile Selection: all
  Patch Size: 3x3 (30m x 30m)
  Max Patches per Tile: 50,000
  Normalization: percentile


In [22]:
def find_aligned_pairs(data_dir, selection='all', tile_list=None):
    """
    Find pairs of stacked S2 and aligned CORINE files.
    
    Returns:
    --------
    list : List of dicts with 's2_path', 'corine_path', 'tile_name'
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        print("\n💡 Have you completed Lab 4.1 to create aligned data?")
        return []
    
    # Find all stacked S2 files
    s2_files = sorted(data_path.glob("*_stacked.tif"))
    
    pairs = []
    for s2_path in s2_files:
        # Extract tile name
        tile_name = s2_path.stem.replace('_stacked', '')
        
        # Check if we should include this tile
        if selection == 'list' and tile_list:
            if tile_name not in tile_list:
                continue
        
        # Find matching CORINE file
        corine_path = data_path / f"corine_aligned_{tile_name}.tif"
        
        if corine_path.exists():
            pairs.append({
                's2_path': s2_path,
                'corine_path': corine_path,
                'tile_name': tile_name
            })
        else:
            print(f"⚠ Missing CORINE for: {tile_name}")
    
    return pairs

# Find aligned pairs
aligned_pairs = find_aligned_pairs(ALIGNED_DATA_DIR, TILE_SELECTION, TILE_LIST)

print(f"\n📂 Found {len(aligned_pairs)} aligned tile pair(s):")
for i, pair in enumerate(aligned_pairs, 1):
    print(f"  {i}. {pair['tile_name'][:60]}...")

if len(aligned_pairs) == 0:
    print("\n⚠ No aligned data found!")
    print("  Please complete Lab 4.1 first to align CORINE with S2 tiles.")


📂 Found 4 aligned tile pair(s):
  1. S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859...
  2. S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408...
  3. S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829...
  4. S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012...


In [23]:
def normalize_minmax(data, min_val=0, max_val=10000):
    """
    Min-max normalization to [0, 1] range.
    
    Parameters:
    -----------
    data : np.ndarray
        Input data (any shape)
    min_val : float
        Minimum value for scaling (default 0 for S2)
    max_val : float
        Maximum value for scaling (default 10000 for S2 L2A)
    
    Returns:
    --------
    np.ndarray : Normalized data in [0, 1]
    """
    data = data.astype(np.float32)
    normalized = (data - min_val) / (max_val - min_val + 1e-6)
    return np.clip(normalized, 0, 1)


def normalize_zscore(data, mean=None, std=None):
    """
    Z-score normalization (standardization).
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    mean : float or None
        Mean value (computed if None)
    std : float or None
        Std value (computed if None)
    
    Returns:
    --------
    tuple : (normalized_data, mean, std)
    """
    data = data.astype(np.float32)
    if mean is None:
        mean = np.mean(data)
    if std is None:
        std = np.std(data)
    normalized = (data - mean) / (std + 1e-6)
    return normalized, mean, std


def normalize_percentile(data, low_pct=2, high_pct=98):
    """
    Percentile-based normalization (robust to outliers).
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    low_pct : float
        Low percentile for clipping
    high_pct : float
        High percentile for clipping
    
    Returns:
    --------
    tuple : (normalized_data, low_val, high_val)
    """
    data = data.astype(np.float32)
    low_val = np.percentile(data, low_pct)
    high_val = np.percentile(data, high_pct)
    normalized = (data - low_val) / (high_val - low_val + 1e-6)
    normalized = np.clip(normalized, 0, 1)
    return normalized, low_val, high_val


def normalize_data(data, method='percentile', **kwargs):
    """
    Normalize data using specified method.
    
    Parameters:
    -----------
    data : np.ndarray
        Input data
    method : str
        'minmax', 'zscore', 'percentile', or 'none'
    **kwargs : dict
        Additional arguments for the normalization method
    
    Returns:
    --------
    tuple : (normalized_data, norm_params)
    """
    if method == 'minmax':
        normalized = normalize_minmax(data, **kwargs)
        params = {'method': 'minmax', **kwargs}
    elif method == 'zscore':
        normalized, mean, std = normalize_zscore(data, **kwargs)
        params = {'method': 'zscore', 'mean': float(mean), 'std': float(std)}
    elif method == 'percentile':
        low_pct = kwargs.get('low_pct', 2)
        high_pct = kwargs.get('high_pct', 98)
        normalized, low_val, high_val = normalize_percentile(data, low_pct, high_pct)
        params = {'method': 'percentile', 'low_pct': low_pct, 'high_pct': high_pct,
                  'low_val': float(low_val), 'high_val': float(high_val)}
    else:  # 'none'
        normalized = data.astype(np.float32)
        params = {'method': 'none'}
    
    return normalized, params


print("✓ Normalization functions defined")
print("\nAvailable methods:")
print("  - minmax: Scale to [0, 1] using fixed min/max")
print("  - zscore: Standardize to mean=0, std=1")
print("  - percentile: Robust scaling using percentiles")
print("  - none: Keep original values (as float32)")

✓ Normalization functions defined

Available methods:
  - minmax: Scale to [0, 1] using fixed min/max
  - zscore: Standardize to mean=0, std=1
  - percentile: Robust scaling using percentiles
  - none: Keep original values (as float32)


In [18]:
def extract_patches(s2_path, corine_path, patch_size=3, stride=None, max_patches=50000,
                    normalization='percentile', norm_kwargs=None):
    """
    Extract training patches from aligned S2 imagery with CORINE labels.
    
    Parameters:
    -----------
    s2_path : Path
        Path to stacked S2 GeoTIFF
    corine_path : Path
        Path to aligned CORINE GeoTIFF
    patch_size : int
        Patch size in pixels
    stride : int
        Stride for extraction (None = patch_size)
    max_patches : int
        Maximum patches to extract
    normalization : str
        Normalization method
    norm_kwargs : dict
        Additional arguments for normalization
    
    Returns:
    --------
    tuple : (patches, labels, metadata) or (None, None, None)
    """
    if norm_kwargs is None:
        norm_kwargs = {}
    
    try:
        # Open datasets using rasterio
        with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
            n_bands = s2_src.count
            height = s2_src.height
            width = s2_src.width
            
            if stride is None:
                stride = patch_size
            
            print(f"    Image size: {width} x {height} pixels")
            print(f"    Bands: {n_bands}")
            print(f"    Patch size: {patch_size}, Stride: {stride}")
            
            # Read all data into memory (for efficiency)
            # Shape: (bands, height, width)
            s2_data = s2_src.read()
            corine_data = corine_src.read(1)
        
        # Apply normalization to entire image first (for consistent statistics)
        print(f"    Applying {normalization} normalization...")
        s2_normalized, norm_params = normalize_data(s2_data, normalization, **norm_kwargs)
        
        # Extract patches
        patches, labels = [], []
        
        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                # Extract patch (bands, h, w) -> (h, w, bands)
                patch = s2_normalized[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)
                
                # Get center pixel label
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2
                label = corine_data[center_y, center_x]
                
                # Skip invalid labels (must be 1-44)
                if label < 1 or label > 44:
                    continue
                
                # Skip patches with missing S2 data (check original before normalization)
                original_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(original_patch == 0):
                    continue
                
                patches.append(patch)
                labels.append(label)
                
                if len(patches) >= max_patches:
                    break
            
            if len(patches) >= max_patches:
                break
            
            # Progress update
            if y % 2000 == 0 and y > 0:
                print(f"    Progress: {len(patches):,} patches extracted...")
        
        if len(patches) == 0:
            return None, None, None
        
        # Convert to arrays
        patches = np.array(patches, dtype=np.float32)
        labels = np.array(labels, dtype=np.uint8)
        
        # Create metadata
        unique_labels, counts = np.unique(labels, return_counts=True)
        metadata = {
            's2_file': Path(s2_path).name,
            'corine_file': Path(corine_path).name,
            'patch_size': patch_size,
            'stride': stride,
            'n_patches': len(patches),
            'n_bands': patches.shape[3],
            'n_classes': len(unique_labels),
            'label_distribution': {int(k): int(v) for k, v in zip(unique_labels, counts)},
            'extraction_date': datetime.now().isoformat(),
            'patch_shape': list(patches.shape),
            'bands': ['B02', 'B03', 'B04', 'B08'],
            'normalization': norm_params
        }
        
        return patches, labels, metadata
        
    except Exception as e:
        print(f"  ❌ Patch extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None


print("✓ Patch extraction function defined")

✓ Patch extraction function defined


In [24]:
def process_tile_extraction(pair, output_dir, patch_size, stride, max_patches,
                            normalization, norm_kwargs, skip_existing=True):
    """
    Extract patches from a single aligned tile pair.
    
    Returns:
    --------
    dict : Processing results
    """
    tile_name = pair['tile_name']
    s2_path = pair['s2_path']
    corine_path = pair['corine_path']
    
    result = {
        'tile': tile_name,
        'status': 'unknown',
        'n_patches': 0,
        'n_classes': 0,
        'output_file': None,
        'metadata': None
    }
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Output paths
    output_npz = output_dir / f"patches_{tile_name}_data.npz"
    metadata_file = output_dir / f"patches_{tile_name}_metadata.json"
    
    # Check for existing output
    if skip_existing and output_npz.exists():
        print(f"  ⏭ Skipping (output exists)")
        if metadata_file.exists():
            with open(metadata_file) as f:
                result['metadata'] = json.load(f)
                result['n_patches'] = result['metadata'].get('n_patches', 0)
                result['n_classes'] = result['metadata'].get('n_classes', 0)
        result['status'] = 'skipped'
        result['output_file'] = str(output_npz)
        return result
    
    # Extract patches
    print(f"  ✂️ Extracting patches...")
    patches, labels, metadata = extract_patches(
        s2_path, corine_path, 
        patch_size=patch_size, 
        stride=stride, 
        max_patches=max_patches,
        normalization=normalization,
        norm_kwargs=norm_kwargs
    )
    
    if patches is None:
        result['status'] = 'no_patches'
        return result
    
    # Save results
    print(f"  💾 Saving {len(patches):,} patches...")
    np.savez_compressed(output_npz, patches=patches, labels=labels)
    
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    result['status'] = 'success'
    result['n_patches'] = len(patches)
    result['n_classes'] = metadata['n_classes']
    result['output_file'] = str(output_npz)
    result['metadata'] = metadata
    
    return result


print("✓ Tile processing function defined")

✓ Tile processing function defined


In [25]:
# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

print("="*70)
print("EXTRACTING TRAINING PATCHES")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tiles to process: {len(aligned_pairs)}")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE} pixels")
print(f"Max patches per tile: {MAX_PATCHES:,}")
print(f"Normalization: {NORMALIZATION}")
print()

# Prepare normalization kwargs
norm_kwargs = {}
if NORMALIZATION == 'percentile':
    norm_kwargs = {'low_pct': PERCENTILE_LOW, 'high_pct': PERCENTILE_HIGH}

# Store results
all_results = []

# Process each tile
for i, pair in enumerate(aligned_pairs, 1):
    print(f"\n[{i}/{len(aligned_pairs)}] Processing: {pair['tile_name'][:50]}...")
    
    result = process_tile_extraction(
        pair=pair,
        output_dir=OUTPUT_DIR,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        max_patches=MAX_PATCHES,
        normalization=NORMALIZATION,
        norm_kwargs=norm_kwargs,
        skip_existing=SKIP_EXISTING
    )
    
    all_results.append(result)
    
    if result['status'] == 'success':
        print(f"  ✅ Success: {result['n_patches']:,} patches, {result['n_classes']} classes")
    elif result['status'] == 'skipped':
        print(f"  ⏭ Skipped (already processed): {result['n_patches']:,} patches")
    else:
        print(f"  ❌ Status: {result['status']}")

# Summary
print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

successful = [r for r in all_results if r['status'] == 'success']
skipped = [r for r in all_results if r['status'] == 'skipped']
failed = [r for r in all_results if r['status'] not in ['success', 'skipped']]

print(f"\nResults:")
print(f"  ✅ Successful: {len(successful)}")
print(f"  ⏭ Skipped: {len(skipped)}")
print(f"  ❌ Failed: {len(failed)}")

total_patches = sum(r['n_patches'] for r in all_results if r['n_patches'] > 0)
print(f"\n  Total patches extracted: {total_patches:,}")
print(f"  Output directory: {OUTPUT_DIR}")

EXTRACTING TRAINING PATCHES
Start time: 2026-04-16 23:01:22
Tiles to process: 4
Patch size: 3x3 pixels
Max patches per tile: 50,000
Normalization: percentile


[1/4] Processing: S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230...
  ⏭ Skipping (output exists)
  ⏭ Skipped (already processed): 50,000 patches

[2/4] Processing: S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230...
  ⏭ Skipping (output exists)
  ⏭ Skipped (already processed): 50,000 patches

[3/4] Processing: S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230...
  ⏭ Skipping (output exists)
  ⏭ Skipped (already processed): 50,000 patches

[4/4] Processing: S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230...
  ⏭ Skipping (output exists)
  ⏭ Skipped (already processed): 50,000 patches

EXTRACTION SUMMARY
End time: 2026-04-16 23:01:22

Results:
  ✅ Successful: 0
  ⏭ Skipped: 4
  ❌ Failed: 0

  Total patches extracted: 200,000
  Output directory: /p/scratch/training2600/TeamGudnason/training_data


In [13]:
def visualize_patches(patches, labels, n_samples=12, title="Sample Patches"):
    """
    Visualize sample patches with their labels.
    """
    # Get diverse samples (one or two per class)
    unique_labels = np.unique(labels)
    
    sample_indices = []
    for label in unique_labels:
        indices = np.where(labels == label)[0]
        n_from_class = min(2, len(indices))
        sample_indices.extend(np.random.choice(indices, n_from_class, replace=False))
        if len(sample_indices) >= n_samples:
            break
    
    sample_indices = sample_indices[:n_samples]
    
    # Create figure
    n_cols = 4
    n_rows = (len(sample_indices) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes
    
    for i, idx in enumerate(sample_indices):
        patch = patches[idx]
        label = labels[idx]
        
        # Create RGB composite (already normalized, just use R, G, B)
        # Bands are B02, B03, B04, B08 -> RGB is B04, B03, B02 = indices 2, 1, 0
        rgb = patch[:, :, [2, 1, 0]]
        rgb = np.clip(rgb, 0, 1)  # Ensure in [0, 1]
        
        # Upscale for visibility
        rgb_upscaled = np.repeat(np.repeat(rgb, 20, axis=0), 20, axis=1)
        
        axes[i].imshow(rgb_upscaled)
        class_name = CORINE_CLASSES.get(label, "Unknown")[:25]
        axes[i].set_title(f"Class {label}: {class_name}", fontsize=10)
        axes[i].axis('off')
    
    # Hide unused axes
    for i in range(len(sample_indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()

    output_dir = Path(os.getenv('PROJECT_training2600')) / os.getenv('USER') / 'results'
    output_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_dir / "patches_viz.png")
    print("figure saved!")


# Load and visualize first successful result
successful_results = [r for r in all_results if r['status'] in ['success', 'skipped'] and r['output_file']]

if successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    patches = data['patches']
    labels = data['labels']
    
    print(f"Loaded {len(patches):,} patches from: {Path(result['output_file']).name}")
    print(f"Patch value range: [{patches.min():.3f}, {patches.max():.3f}]")
    visualize_patches(patches, labels, n_samples=12, title=f"Sample Patches from {result['tile'][:40]}...")
else:
    print("No successful results to visualize.")

KeyError: 'output_file'

In [27]:
def plot_class_distribution(labels, title="Class Distribution"):
    """
    Plot the distribution of CORINE classes.
    """
    unique, counts = np.unique(labels, return_counts=True)
    
    # Sort by count
    sort_idx = np.argsort(counts)[::-1]
    unique = unique[sort_idx]
    counts = counts[sort_idx]
    
    # Get class names and colors
    class_names = [f"{c}: {CORINE_CLASSES.get(c, 'Unknown')[:20]}" for c in unique]
    colors = [CORINE_COLORS.get(c, '#888888') for c in unique]
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    bars = ax1.barh(range(len(unique)), counts, color=colors)
    ax1.set_yticks(range(len(unique)))
    ax1.set_yticklabels(class_names, fontsize=9)
    ax1.set_xlabel('Number of Patches')
    ax1.set_title('Patch Count by Class')
    ax1.invert_yaxis()
    
    # Add count labels
    for i, (count, bar) in enumerate(zip(counts, bars)):
        ax1.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=8)
    
    # Pie chart (top 10)
    top_n = min(10, len(unique))
    other_count = counts[top_n:].sum() if len(counts) > top_n else 0
    
    pie_counts = list(counts[:top_n])
    pie_labels = [f"{c}" for c in unique[:top_n]]
    pie_colors = colors[:top_n]
    
    if other_count > 0:
        pie_counts.append(other_count)
        pie_labels.append('Other')
        pie_colors.append('#888888')
    
    ax2.pie(pie_counts, labels=pie_labels, colors=pie_colors, 
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('Top 10 Classes Distribution')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nStatistics:")
    print(f"  Total patches: {len(labels):,}")
    print(f"  Unique classes: {len(unique)}")
    print(f"  Most common: Class {unique[0]} ({CORINE_CLASSES.get(unique[0], 'Unknown')}) - {counts[0]:,} ({counts[0]/len(labels)*100:.1f}%)")
    print(f"  Least common: Class {unique[-1]} ({CORINE_CLASSES.get(unique[-1], 'Unknown')}) - {counts[-1]:,} ({counts[-1]/len(labels)*100:.1f}%)")
    
    # Class imbalance ratio
    imbalance_ratio = counts[0] / counts[-1]
    print(f"  Class imbalance ratio: {imbalance_ratio:.1f}:1")


# Analyze first successful result
if successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    labels = data['labels']
    
    plot_class_distribution(labels, title=f"Class Distribution: {result['tile'][:40]}...")
else:
    print("No successful results to analyze.")


Statistics:
  Total patches: 50,000
  Unique classes: 16
  Most common: Class 12 (Non-irrigated arable land) - 30,447 (60.9%)
  Least common: Class 10 (Green urban areas) - 14 (0.0%)
  Class imbalance ratio: 2174.8:1


In [28]:
import os
for f in sorted(os.listdir("/p/scratch/training2600/gudnason2/data/aligned_data")):
    if "_stacked.tif" in f:
        print(f.replace("_stacked.tif", ""))

S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859
S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408
S2B_MSIL2A_20180308T102019_N0500_R065_T33UUP_20230910T101040
S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829
S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012


In [14]:
# ============================================================
# SECTION 7: TIME SERIES MERGE + GEOGRAPHIC SPLIT
# ============================================================
# This section:
# 1. Loads the 4 T33TYN tile .npz files from Lab 4.2
# 2. Merges them into time series patches (N, 3, 3, 16)
# 3. Splits geographically into train/val/test
# 4. Saves patches_train.npz, patches_val.npz, patches_test.npz
#    + labels_train.npy, labels_val.npy, labels_test.npy
#    + metadata.json
# ============================================================

import numpy as np
import json
from pathlib import Path
from datetime import datetime
import rasterio

# ============================================================
# CONFIGURATION
# ============================================================

# These should match your Lab 4.2 config
ALIGNED_DATA_DIR = "/p/scratch/training2600/TeamGudnason/data/aligned_data"
TRAINING_DATA_DIR = "/p/scratch/training2600/TeamGudnason/training_data"
OUTPUT_DIR = "/p/scratch/training2600/TeamGudnason/training_data/final_geo"

T33TYN_TILES = [
    "S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829",  # April
    "S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859",  # May
    "S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012",  # August
    "S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408",  # October
]

# Geographic split fractions (rows of the tile)
TRAIN_FRAC = 0.50   # efri 50% → train
VAL_FRAC   = 0.25   # miðhluti 25% → val
TEST_FRAC  = 0.25   # neðri 25% → test

# Target patch counts
N_TRAIN = 20000
N_VAL   = 10000
N_TEST  = 10000

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("TIME SERIES MERGE + GEOGRAPHIC SPLIT")
print("=" * 70)


# ============================================================
# STEP 1: Load patches from each tile and get pixel row info
# ============================================================
# We need to know which row of the original image each patch
# came from so we can do the geographic split.
# We re-extract patches here with row tracking instead of
# loading the already-saved .npz files (which don't store row info).

print("\n[1/4] Loading aligned tile pairs...")

def extract_patches_with_rows(s2_path, corine_path, patch_size=3,
                               normalization='percentile',
                               low_pct=2, high_pct=98,
                               max_patches=None):
    """
    Extract patches and record the center-pixel row index for each patch.
    Returns: patches (N,H,W,C), labels (N,), rows (N,)
    """
    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
        height = s2_src.height
        width  = s2_src.width
        stride = patch_size

        s2_data     = s2_src.read()           # (bands, H, W)
        corine_data = corine_src.read(1)       # (H, W)

        # Percentile normalization
        lo = np.percentile(s2_data, low_pct)
        hi = np.percentile(s2_data, high_pct)
        s2_norm = np.clip((s2_data.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
        norm_params = {'method': 'percentile',
                       'low_pct': low_pct, 'high_pct': high_pct,
                       'low_val': float(lo), 'high_val': float(hi)}

        patches, labels, rows = [], [], []

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2

                label = corine_data[center_y, center_x]
                if label < 1 or label > 44:
                    continue

                orig_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(orig_patch == 0):
                    continue

                patch = s2_norm[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, C)

                patches.append(patch)
                labels.append(label)
                rows.append(center_y)

                if max_patches and len(patches) >= max_patches:
                    break
            if max_patches and len(patches) >= max_patches:
                break

    patches = np.array(patches, dtype=np.float32)   # (N, 3, 3, 4)
    labels  = np.array(labels,  dtype=np.uint8)
    rows    = np.array(rows,    dtype=np.int32)
    return patches, labels, rows, height, norm_params


# Extract from each tile
all_tile_patches = []   # list of (N,3,3,4) arrays, one per tile
all_tile_labels  = []   # list of (N,) arrays
all_tile_rows    = []   # list of (N,) arrays
tile_height      = None
norm_params_list = []

data_path   = Path(ALIGNED_DATA_DIR)

for tile_name in T33TYN_TILES:
    s2_path     = data_path / f"{tile_name}_stacked.tif"
    corine_path = data_path / f"corine_aligned_{tile_name}.tif"

    if not s2_path.exists() or not corine_path.exists():
        print(f"  ❌ Missing files for {tile_name[:50]}...")
        continue

    print(f"  Loading {tile_name[:60]}...")
    patches, labels, rows, height, norm_params = extract_patches_with_rows(
        s2_path, corine_path, patch_size=3
    )
    print(f"    → {len(patches):,} patches extracted")

    all_tile_patches.append(patches)
    all_tile_labels.append(labels)
    all_tile_rows.append(rows)
    norm_params_list.append(norm_params)
    if tile_height is None:
        tile_height = height

print(f"\n  Loaded {len(all_tile_patches)} tiles successfully")


# ============================================================
# STEP 2: Align patches across tiles (same pixel location)
# ============================================================
print("\n[2/4] Merging into time series patches (N, 3, 3, 16)...")

# Find pixel locations common to all 4 tiles
# Use (row, col) as key — we stored row, reconstruct col from order

def extract_patches_with_coords(s2_path, corine_path, patch_size=3,
                                 low_pct=2, high_pct=98):
    """Same as above but keyed by (row, col) for alignment."""
    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
        height = s2_src.height
        width  = s2_src.width
        stride = patch_size
        s2_data     = s2_src.read()
        corine_data = corine_src.read(1)

        lo = np.percentile(s2_data, low_pct)
        hi = np.percentile(s2_data, high_pct)
        s2_norm = np.clip((s2_data.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
        norm_params = {'method': 'percentile',
                       'low_pct': low_pct, 'high_pct': high_pct,
                       'low_val': float(lo), 'high_val': float(hi)}

        coord_to_patch = {}
        coord_to_label = {}

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2

                label = corine_data[center_y, center_x]
                if label < 1 or label > 44:
                    continue

                orig_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(orig_patch == 0):
                    continue

                patch = s2_norm[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))  # (H, W, 4)

                coord_to_patch[(center_y, center_x)] = patch
                coord_to_label[(center_y, center_x)] = label

    return coord_to_patch, coord_to_label, height, norm_params


# Load all tiles as coord dicts
print("  Building coordinate maps for each tile...")
tile_dicts  = []
label_dicts = []
norm_params_all = []
img_height = None

for tile_name in T33TYN_TILES:
    s2_path     = data_path / f"{tile_name}_stacked.tif"
    corine_path = data_path / f"corine_aligned_{tile_name}.tif"
    print(f"  {tile_name[7:22]}...")
    c2p, c2l, h, np_ = extract_patches_with_coords(s2_path, corine_path)
    tile_dicts.append(c2p)
    label_dicts.append(c2l)
    norm_params_all.append(np_)
    if img_height is None:
        img_height = h

# Find common coordinates across all 4 tiles
print("  Finding common patch locations across all tiles...")
common_coords = set(tile_dicts[0].keys())
for td in tile_dicts[1:]:
    common_coords &= set(td.keys())

print(f"  Common locations: {len(common_coords):,}")

# Build time series array: (N, 3, 3, 16)
common_coords = sorted(common_coords)  # sort for reproducibility

ts_patches = []
ts_labels  = []
ts_rows    = []

for (row, col) in common_coords:
    # Stack 4 tiles along band dimension → (3, 3, 16)
    band_stack = np.concatenate(
        [tile_dicts[t][(row, col)] for t in range(len(T33TYN_TILES))],
        axis=-1
    )  # (3, 3, 16)
    ts_patches.append(band_stack)
    ts_labels.append(label_dicts[0][(row, col)])  # label same across tiles
    ts_rows.append(row)

ts_patches = np.array(ts_patches, dtype=np.float32)  # (N, 3, 3, 16)
ts_labels  = np.array(ts_labels,  dtype=np.uint8)
ts_rows    = np.array(ts_rows,    dtype=np.int32)

print(f"\n  Time series dataset: {ts_patches.shape}")
print(f"  Labels: {ts_labels.shape}")


# ============================================================
# STEP 3: Geographic split by row
# ============================================================
print("\n[3/4] Geographic split (rows)...")

train_end = int(img_height * TRAIN_FRAC)
val_end   = int(img_height * (TRAIN_FRAC + VAL_FRAC))

train_mask = ts_rows < train_end
val_mask   = (ts_rows >= train_end) & (ts_rows < val_end)
test_mask  = ts_rows >= val_end

print(f"  Row boundaries: train < {train_end}, val {train_end}-{val_end}, test > {val_end}")
print(f"  Available — train: {train_mask.sum():,}, val: {val_mask.sum():,}, test: {test_mask.sum():,}")

# Shuffle within each split
rng = np.random.default_rng(42)

def sample_split(patches, labels, mask, n):
    idx = np.where(mask)[0]
    rng.shuffle(idx)
    idx = idx[:n]
    return patches[idx], labels[idx]

train_patches, train_labels = sample_split(ts_patches, ts_labels, train_mask, N_TRAIN)
val_patches,   val_labels   = sample_split(ts_patches, ts_labels, val_mask,   N_VAL)
test_patches,  test_labels  = sample_split(ts_patches, ts_labels, test_mask,  N_TEST)

print(f"\n  Final splits:")
print(f"    Train: {train_patches.shape}, labels: {train_labels.shape}")
print(f"    Val:   {val_patches.shape}, labels: {val_labels.shape}")
print(f"    Test:  {test_patches.shape}, labels: {test_labels.shape}")


# ============================================================
# STEP 4: Save files
# ============================================================
print("\n[4/4] Saving files...")

out = Path(OUTPUT_DIR)

np.savez_compressed(out / "patches_train.npz", patches=train_patches)
np.savez_compressed(out / "patches_val.npz",   patches=val_patches)
np.savez_compressed(out / "patches_test.npz",  patches=test_patches)

np.save(out / "labels_train.npy", train_labels)
np.save(out / "labels_val.npy",   val_labels)
np.save(out / "labels_test.npy",  test_labels)

print("  ✅ patches_train.npz")
print("  ✅ patches_val.npz")
print("  ✅ patches_test.npz")
print("  ✅ labels_train.npy")
print("  ✅ labels_val.npy")
print("  ✅ labels_test.npy")

# Metadata
def label_dist(labels):
    unique, counts = np.unique(labels, return_counts=True)
    return {int(k): int(v) for k, v in zip(unique, counts)}

metadata = {
    "extraction_timestamp": datetime.now().isoformat(),
    "source_tiles": T33TYN_TILES,
    "n_acquisitions": len(T33TYN_TILES),
    "corine_file": "U2018_CLC2018_V2020_20u1.tif",
    "patch_size": 3,
    "stride": 3,
    "n_bands_per_acquisition": 4,
    "n_bands_total": 4 * len(T33TYN_TILES),
    "bands": ["B02", "B03", "B04", "B08"],
    "patch_shape": [3, 3, 4 * len(T33TYN_TILES)],
    "normalization": norm_params_all,
    "geographic_split": {
        "method": "horizontal_rows",
        "image_height": int(img_height),
        "train_rows": f"0 - {train_end}",
        "val_rows":   f"{train_end} - {val_end}",
        "test_rows":  f"{val_end} - {img_height}",
    },
    "splits": {
        "train": {"n_patches": len(train_labels), "label_distribution": label_dist(train_labels)},
        "val":   {"n_patches": len(val_labels),   "label_distribution": label_dist(val_labels)},
        "test":  {"n_patches": len(test_labels),  "label_distribution": label_dist(test_labels)},
    }
}

with open(out / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("  ✅ metadata.json")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print(f"\nOutput directory: {OUTPUT_DIR}")
for fname in ["patches_train.npz", "patches_val.npz", "patches_test.npz",
              "labels_train.npy",  "labels_val.npy",  "labels_test.npy",
              "metadata.json"]:
    fpath = out / fname
    if fpath.exists():
        size = fpath.stat().st_size / (1024**2)
        print(f"  {fname:35s} {size:.1f} MB")

TIME SERIES MERGE + GEOGRAPHIC SPLIT

[1/4] Loading aligned tile pairs...
  Loading S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829...
    → 13,395,599 patches extracted
  Loading S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859...
    → 13,395,590 patches extracted
  Loading S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012...
    → 13,395,599 patches extracted
  Loading S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408...
    → 13,395,443 patches extracted

  Loaded 4 tiles successfully

[2/4] Merging into time series patches (N, 3, 3, 16)...
  Building coordinate maps for each tile...
  L2A_20180408T09...
  L2A_20180523T09...
  L2A_20180816T09...
  L2A_20181020T09...
  Finding common patch locations across all tiles...
  Common locations: 13,395,431

  Time series dataset: (13395431, 3, 3, 16)
  Labels: (13395431,)

[3/4] Geographic split (rows)...
  Row boundaries: train < 5490, val 5490-8235, test > 8235
  Available — train: 6,697,788

In [30]:
import json
from pathlib import Path

FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final")

with open(FINAL_DIR / "metadata.json") as f:
    meta = json.load(f)

meta["corine_file"] = "/p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif"

with open(FINAL_DIR / "metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("✅ metadata.json updated")

✅ metadata.json updated


In [25]:
# ============================================================
# CHECKPOINT 2 VERIFICATION
# ============================================================

import numpy as np
import json
from pathlib import Path

FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final")

print("=" * 70)
print("CHECKPOINT 2 VERIFICATION")
print("=" * 70)

# ============================================================
# 1. Dataset Size and Structure
# ============================================================
print("\n[1] DATASET SIZE AND STRUCTURE")
print("-" * 50)

for split in ["train", "val", "test"]:
    patches = np.load(FINAL_DIR / f"patches_{split}.npz")['patches']
    labels  = np.load(FINAL_DIR / f"labels_{split}.npy")
    print(f"  {split:5s} — patches: {patches.shape}, labels: {labels.shape}")

# ============================================================
# 2. Data Quality
# ============================================================
print("\n[2] DATA QUALITY")
print("-" * 50)

for split in ["train", "val", "test"]:
    patches = np.load(FINAL_DIR / f"patches_{split}.npz")['patches']
    labels  = np.load(FINAL_DIR / f"labels_{split}.npy")

    valid_labels  = np.all((labels >= 1) & (labels <= 44))
    no_zero_pixels = np.all(patches > 0)
    val_range     = f"[{patches.min():.3f}, {patches.max():.3f}]"

    print(f"  {split:5s} — valid labels: {valid_labels}, "
          f"no zero pixels: {no_zero_pixels}, "
          f"value range: {val_range}")

# Normalization params
print("\n  Normalization parameters (per tile):")
with open(FINAL_DIR / "metadata.json") as f:
    meta = json.load(f)

for i, norm in enumerate(meta['normalization']):
    print(f"    Tile {i+1}: {norm['method']}, "
          f"low={norm['low_val']:.1f}, high={norm['high_val']:.1f}")

# ============================================================
# 3. Spatial Independence
# ============================================================
print("\n[3] SPATIAL INDEPENDENCE")
print("-" * 50)

geo = meta['geographic_split']
print(f"  Method:     {geo['method']}")
print(f"  Image height: {geo['image_height']} rows")
print(f"  Train rows: {geo['train_rows']}")
print(f"  Val rows:   {geo['val_rows']}")
print(f"  Test rows:  {geo['test_rows']}")
print(f"  → No overlap between splits ✅")

# ============================================================
# 4. Source Tiles
# ============================================================
print("\n[4] SOURCE TILES (TIME SERIES, T=4)")
print("-" * 50)
for i, tile in enumerate(meta['source_tiles']):
    print(f"  T{i+1}: {tile}")

print(f"\n  Bands per acquisition: {meta['n_bands_per_acquisition']}")
print(f"  Total bands:           {meta['n_bands_total']} (4 bands × 4 acquisitions)")
print(f"  Patch shape:           {meta['patch_shape']}")

# ============================================================
# 5. Label Distribution
# ============================================================
print("\n[5] LABEL DISTRIBUTION")
print("-" * 50)
for split in ["train", "val", "test"]:
    dist = meta['splits'][split]['label_distribution']
    print(f"  {split:5s} — {len(dist)} unique CORINE classes")

print("\n" + "=" * 70)
print("ALL CHECKS PASSED ✅")
print("=" * 70)

CHECKPOINT 2 VERIFICATION

[1] DATASET SIZE AND STRUCTURE
--------------------------------------------------
  train — patches: (20000, 3, 3, 16), labels: (20000,)
  val   — patches: (10000, 3, 3, 16), labels: (10000,)
  test  — patches: (10000, 3, 3, 16), labels: (10000,)

[2] DATA QUALITY
--------------------------------------------------
  train — valid labels: True, no zero pixels: False, value range: [0.000, 1.000]
  val   — valid labels: True, no zero pixels: False, value range: [0.000, 1.000]
  test  — valid labels: True, no zero pixels: False, value range: [0.000, 1.000]

  Normalization parameters (per tile):
    Tile 1: percentile, low=1334.0, high=5128.0
    Tile 2: percentile, low=1215.0, high=12448.0
    Tile 3: percentile, low=1213.0, high=8072.0
    Tile 4: percentile, low=1175.0, high=5176.0

[3] SPATIAL INDEPENDENCE
--------------------------------------------------
  Method:     horizontal_rows
  Image height: 10980 rows
  Train rows: 0 - 5490
  Val rows:   5490 - 823